# Export Invoice Summary — Data Processing Pipeline

This notebook pulls invoice data (Woven and Sweater company groups) from the
reporting database, cleans and standardizes it, and produces the following
deliverables:

1. **EIS.xlsx** — the master, cleaned Export Invoice Summary
2. **RealizeReports/** — Realize Pending report, split by owner/category
3. **OnBoardPending.xlsx** — invoices awaiting on-board status
4. **ExFactoryPending.xlsx** — invoices awaiting ex-factory shipment
5. **BankSubmitPending.xlsx** — invoices awaiting bank submission

## 1. Setup & Configuration

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import pyodbc
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# Output locations
# ----------------------------------------------------------------------
OUTPUT_DIR = "OUTPUT"
REALIZE_REPORTS_DIR = os.path.join(OUTPUT_DIR, "RealizeReports")

for directory in (OUTPUT_DIR, REALIZE_REPORTS_DIR):
    os.makedirs(directory, exist_ok=True)

## 2. Database Connection

In [2]:
load_dotenv()

DB_SERVER = os.getenv("DB_SERVER")
DB_PORT = os.getenv("DB_PORT")
DB_DATABASE = os.getenv("DB_DATABASE")
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")

CONN_STR = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={DB_SERVER},{DB_PORT};"
    f"DATABASE={DB_DATABASE};"
    f"UID={DB_USERNAME};"
    f"PWD={DB_PASSWORD};"
    "TrustServerCertificate=yes;"  # stability improvement
    "Encrypt=no;"                  # stability improvement
    "MARS_Connection=yes;"         # required for multi-result stored procedures
)


def get_connection():
    """Open a SQL Server connection using the configured credentials.

    Returns
    -------
    pyodbc.Connection | None
        An open, autocommitting connection, or ``None`` if the connection
        attempt failed (the error is printed for diagnostics).
    """
    try:
        conn = pyodbc.connect(CONN_STR, timeout=10, autocommit=True)
        conn.timeout = 30  # query execution timeout (seconds)
        print("Connected successfully")
        return conn
    except pyodbc.Error as e:
        print("Connection failed:", e)
        return None

In [3]:
conn = get_connection()

if conn is None:
    print("CRITICAL: Connection could not be established. Check DB_SERVER and DB_PASSWORD.")

Connected successfully


## 3. Extract Invoice Data

Data is pulled separately for the two company groups (Woven = 1, Sweater = 2)
via the `Report_ExportInvoiceSummary` stored procedure, then combined into a
single working `DataFrame`.

In [4]:
def fetch_invoice_summary(connection, company_group_id, from_date, to_date,
                           report_type_id=0, fiscal_year_id=0, is_realize=0):
    """Run the Report_ExportInvoiceSummary stored procedure and return the result.

    Parameters
    ----------
    connection : pyodbc.Connection
        An open database connection.
    company_group_id : int
        1 = Woven, 2 = Sweater.
    from_date, to_date : str
        Date range in 'YYYY-MM-DD' format.

    Returns
    -------
    pandas.DataFrame
    """
    query = f"""
    SET NOCOUNT ON;
    EXEC Report_ExportInvoiceSummary
         @ReportTypeID   = {report_type_id},
         @FiscalYearID   = {fiscal_year_id},
         @FromDate       = '{from_date}',
         @ToDate         = '{to_date}',
         @CompanyGroupID = {company_group_id},
         @isRealize      = {is_realize}
    """
    return pd.read_sql_query(query, connection)

In [5]:
if conn is not None:
    # Woven = CompanyGroupID 1
    woven = fetch_invoice_summary(
        conn, company_group_id=1, from_date="2020-01-01", to_date="2026-12-31"
    )
    print(f"Woven Data Shape   : {woven.shape}")

    # Sweater = CompanyGroupID 2
    sweater = fetch_invoice_summary(
        conn, company_group_id=2, from_date="2020-01-01", to_date="2026-12-31"
    )
    print(f"Sweater Data Shape : {sweater.shape}")
else:
    raise RuntimeError("Connection is None. Check your credentials/server.")

Woven Data Shape   : (50390, 44)
Sweater Data Shape : (1613, 44)


In [6]:
df = pd.concat([woven, sweater], ignore_index=False)
print(f"Combined Data Shape : {df.shape}")
df.sample(5)

Combined Data Shape : (52003, 44)


,LCFactory,LCNo,LCValue,LCQuantity,PaymentTerm,SalesInvoice,ExpNo,ExpDate,SalesInvoiceDate,Buyer,...,BankRefNo,BankSubmitDate,IsFDBC,BankShortName,ShippingModeName,InvoiceStatus,RealizeNO,RealizeDate,RealizeValue,IsRealize
18907,MGSL,MGSL/H&M-Mens DBL-S.0,11664021.68,2755553.0,EOM+63,2410730,00001689-012363-2024,2024-07-18,2024-07-18 11:43:18.660,H&M,...,FDC-2969-24 (DBBL PART),2024-09-04,Yes,DBBL,Sea,Incentive Pending,DR-09-24-005,04-Sep-2024,899.36,Yes
17257,MGSL,MGSL/H&M-MENS-S.0,7963192.82,1672612.0,EOM+63,2408995,00000042-014213-2024,2024-06-09,2024-06-09 00:00:00.000,H&M,...,0784/24 (AGRANI PART),2024-06-27,Yes,ABL,Sea,Incentive Pending,DR-07-24-006,04-Jul-2024,2488.60,Yes
18588,MGSL,MGSL/WOMEN SHIRT S.0,3305216.33,677927.0,EOM+63,2410397,00000042-016005-2024,2024-07-11,2024-07-11 12:47:28.700,H&M,...,FDBC-880/24,2024-07-28,Yes,ABL,Sea,Incentive Pending,DR-07-24-023,28-Jul-2024,610.17,Yes
48553,MGSL,MGSL/Women Shirt S.4,6491586.16,1268650.0,EOM+63,2608502,00001689-018301-2026,2026-07-23,2026-07-23 15:54:37.890,H&M,...,NaN,NaT,No,DBBL,Sea,Bank Submit Pending,NaN,NaN,NaN,No
11333,MGSL,MGSL/H&M-MENS-S.09,4542005.37,1244813.0,EOM+63,2402777,042/004686/24,2024-02-15,2024-02-14 00:00:00.000,H&M,...,580/24,2024-05-06,Yes,ABL,Sea,Incentive Pending,DR-05-24-049,08-May-2024,969.60,Yes


In [7]:
df.columns

Index(['LCFactory', 'LCNo', 'LCValue', 'LCQuantity', 'PaymentTerm',
       'SalesInvoice', 'ExpNo', 'ExpDate', 'SalesInvoiceDate', 'Buyer',
       'OrderNo', 'Merchandiser', 'PortName', 'OrderQty', 'Dozen', 'ctnQty',
       'TotalFOB', 'DiscountAmountUSD', 'CommissionAmountFC', 'ExFactoryQty',
       'InvExFactoryDate', 'ExfactoryDate', 'ExFactoryNo', 'ExFactory',
       'OnBoardDate', 'DueDate', 'FVesselNo', 'MVesselNo', 'ShippingBillNo',
       'SBillDate', 'BLNo', 'BLDate', 'FCRNo', 'FCRDate', 'BankRefNo',
       'BankSubmitDate', 'IsFDBC', 'BankShortName', 'ShippingModeName',
       'InvoiceStatus', 'RealizeNO', 'RealizeDate', 'RealizeValue',
       'IsRealize'],
      dtype='str')

In [8]:
############ Shuffle the df Dataframe ############
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.sample(5)

,LCFactory,LCNo,LCValue,LCQuantity,PaymentTerm,SalesInvoice,ExpNo,ExpDate,SalesInvoiceDate,Buyer,...,BankRefNo,BankSubmitDate,IsFDBC,BankShortName,ShippingModeName,InvoiceStatus,RealizeNO,RealizeDate,RealizeValue,IsRealize
138,MGSL,MGSL/Women Shirt S.2,2995246.39,619756.0,EOM+63,2506837,00001689-015101-2025,2025-05-18,2025-05-17 00:00:00.000,H&M,...,FDC-2908-25,2025-08-06,Yes,DBBL,Sea,Incentive Pending,DR-08-25-008,08-Aug-2025,970.20,Yes
32808,MGSL,MGSL/H&M Kids S2,11273220.28,3616545.0,EOM 63,2507851,00001689-017540-2025,2025-06-19,2025-06-19 14:51:42.470,H&M,...,FDC-2416-25,2025-06-26,Yes,DBBL,Sea,Incentive Pending,DR-06-25-032,26-Jun-2025,3119.97,Yes
7427,MGSL,MGSL/PRINCETON SHIRT-8757 S.0,2740870.51,648692.0,EOM+63,2409404,00000042-014796-2024,2024-06-23,2024-06-23 16:21:39.617,H&M,...,FDBC-804/24 (AGRANI PART),2024-07-03,Yes,ABL,Sea,Incentive Pending,DR-07-24-007,04-Jul-2024,312.48,Yes
22089,MGL,Primark Lingerie BA 25,6905193.20,17834350.0,30 Days,2606304,00000947-000963-2026,2026-06-06,2026-06-06 00:00:00.000,Primark,...,FDBC-0356-26,2026-06-25,Yes,NBL,Sea,Incentive Pending,DR-06-26-045,25-Jun-2026,2480.00,Yes
310,MGSL,MGSL/H&M-MENS-S.0,7963192.82,1672612.0,EOM+63,2405891,042/009904/24,2024-04-21,2024-04-20 00:00:00.000,H&M,...,920/24,2024-08-08,Yes,ABL,Sea,Incentive Pending,DR-08-24-006,11-Aug-2024,227.92,Yes


In [9]:
if "RealizeDate" in df.columns:
    df["RealizeDate"] = pd.to_datetime(df["RealizeDate"], errors="coerce")

# Keep RealizeDate as datetime dtype so Excel can treat it as a real date/time value
# Avoid converting it to a formatted string here.
df[["RealizeDate"]].head()

,RealizeDate
0,2026-01-08
1,2024-10-08
2,2024-05-08
3,2024-03-06
4,2025-06-04


In [10]:
# Normalize common date columns to datetime and treat year 1900 as missing
date_cols = [
    "Ex_Factory_Date", "Invoice_Date", "Realized_Date",
    "Shipping_Bill_Date", "BL_Date", "FCR_Date",
    "Bank_Submit_Date", "OnBoard_Date", "Exp_Date",
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        # Replace any parsed dates in the year 1900 with NaT so Excel shows a blank
        mask1900 = df[col].notna() & (df[col].dt.year == 1900)
        if mask1900.any():
            df.loc[mask1900, col] = pd.NaT

df.columns

Index(['LCFactory', 'LCNo', 'LCValue', 'LCQuantity', 'PaymentTerm',
       'SalesInvoice', 'ExpNo', 'ExpDate', 'SalesInvoiceDate', 'Buyer',
       'OrderNo', 'Merchandiser', 'PortName', 'OrderQty', 'Dozen', 'ctnQty',
       'TotalFOB', 'DiscountAmountUSD', 'CommissionAmountFC', 'ExFactoryQty',
       'InvExFactoryDate', 'ExfactoryDate', 'ExFactoryNo', 'ExFactory',
       'OnBoardDate', 'DueDate', 'FVesselNo', 'MVesselNo', 'ShippingBillNo',
       'SBillDate', 'BLNo', 'BLDate', 'FCRNo', 'FCRDate', 'BankRefNo',
       'BankSubmitDate', 'IsFDBC', 'BankShortName', 'ShippingModeName',
       'InvoiceStatus', 'RealizeNO', 'RealizeDate', 'RealizeValue',
       'IsRealize'],
      dtype='str')

In [11]:
rename_map = {
    "LCFactory": "LC_Factory",
    "LCNo": "LC_No",
    "LCValue": "LC_Value",
    "LCQuantity": "LC_Qty",
    "PaymentTerm": "Payment_Term",
    "SalesInvoice": "Invoice_No",
    "Buyer": "Buyer",
    "ExpNo": "Exp_No",
    "ExpDate": "Exp_Date",
    "SalesInvoiceDate": "Invoice_Date",
    "OrderNo": "Order_No",
    "PortName": "Destination",
    "OrderQty": "Invoice_Qty",
    "Dozen": "Dozen_Qty",
    "ctnQty": "Carton_Qty",
    "TotalFOB": "Invoice_Value",
    "DiscountAmountUSD": "Discount",
    "CommissionAmountFC": "Commission",
    "ExFactoryQty": "Ex_Factory_Qty",
    "ExfactoryDate": "Ex_Factory_Date",
    "ExFactoryNo": "Ex_Factory_No",
    "ExFactory": "Ex_Factory",
    "OnBoardDate": "OnBoard_Date",
    "DueDate": "Due_Date",
    "FVesselNo": "Feeder_Vessel_No",
    "MVesselNo": "Mother_Vessel_No",
    "ShippingBillNo": "Shipping_Bill_No",
    "SBillDate": "Shipping_Bill_Date",
    "BLNo": "BL_No",
    "BLDate": "BL_Date",
    "FCRNo": "FCR_No",
    "FCRDate": "FCR_Date",
    "BankRefNo": "Bank_Ref_No",
    "BankSubmitDate": "Bank_Submit_Date",
    "IsFDBC": "Is_FDBC?",
    "BankShortName": "Bank",
    "ShippingModeName": "Shipping_Mode",
    "InvoiceStatus": "Invoice_Status",
    "RealizeNO": "Realized_No",
    "RealizeDate": "Realized_Date",
    "RealizeValue": "Realized_Value",
    "IsRealize": "Is_Realized?"
}

# Keep only existing columns
existing_rename_map = {k: v for k, v in rename_map.items() if k in df.columns}

# Rename columns
df.rename(columns=existing_rename_map, inplace=True)

# Display renamed columns
print(f"Renamed {len(existing_rename_map)} columns:")
for old, new in existing_rename_map.items():
    print(f"{old}  -->  {new}")

print("\nFinal columns:")
print(df.columns.tolist())

Renamed 42 columns:
LCFactory  -->  LC_Factory
LCNo  -->  LC_No
LCValue  -->  LC_Value
LCQuantity  -->  LC_Qty
PaymentTerm  -->  Payment_Term
SalesInvoice  -->  Invoice_No
Buyer  -->  Buyer
ExpNo  -->  Exp_No
ExpDate  -->  Exp_Date
SalesInvoiceDate  -->  Invoice_Date
OrderNo  -->  Order_No
PortName  -->  Destination
OrderQty  -->  Invoice_Qty
Dozen  -->  Dozen_Qty
ctnQty  -->  Carton_Qty
TotalFOB  -->  Invoice_Value
DiscountAmountUSD  -->  Discount
CommissionAmountFC  -->  Commission
ExFactoryQty  -->  Ex_Factory_Qty
ExfactoryDate  -->  Ex_Factory_Date
ExFactoryNo  -->  Ex_Factory_No
ExFactory  -->  Ex_Factory
OnBoardDate  -->  OnBoard_Date
DueDate  -->  Due_Date
FVesselNo  -->  Feeder_Vessel_No
MVesselNo  -->  Mother_Vessel_No
ShippingBillNo  -->  Shipping_Bill_No
SBillDate  -->  Shipping_Bill_Date
BLNo  -->  BL_No
BLDate  -->  BL_Date
FCRNo  -->  FCR_No
FCRDate  -->  FCR_Date
BankRefNo  -->  Bank_Ref_No
BankSubmitDate  -->  Bank_Submit_Date
IsFDBC  -->  Is_FDBC?
BankShortName  -->  B

In [12]:
# Factories MFSL / MGKFL belong to the Sweater category; everything else is Woven
df["Category"] = np.where(df["LC_Factory"].isin(["MFSL", "MGKFL"]), "Sweater", "Woven")

In [13]:

# Reorder dataframe columns explicitly by listing the desired sequence
# Replace the column names below with the order you want.

desired_columns = [
    "LC_Factory",
    "LC_No",
    "LC_Value",
    "LC_Qty",
    "Payment_Term",

    "Category",
    "Buyer",
    "Invoice_No",
    "Invoice_Date",
    "Invoice_Qty",
    "Invoice_Value",

    "Exp_No",
    "Exp_Date",

    "Ex_Factory",
    "Ex_Factory_Qty",
    "Ex_Factory_Date",
    "Ex_Factory_No",

    "OnBoard_Date",
    "Due_Date",

    "Discount",
    "Commission",
    
    "Shipping_Bill_No",
    "Shipping_Bill_Date",
    "BL_No",
    "BL_Date",
    "FCR_No",
    "FCR_Date",
    "Shipping_Mode",

    "Feeder_Vessel_No",
    "Mother_Vessel_No",
    "Destination",

    "Bank",
    "Bank_Ref_No",
    "Bank_Submit_Date",
    "Is_FDBC?",

    "Invoice_Status",

    "Realized_No", 
    "Realized_Date",
    "Realized_Value",
    "Is_Realized?",

    "Order_No",

]

# Keep only columns that already exist in the dataframe
desired_columns = [col for col in desired_columns if col in df.columns]
# Final ordered dataframe
df = df[desired_columns]

# Add serial number column at the beginning
df.insert(0, "SL", range(1, len(df) + 1))

print("Reordered columns:")
print(df.columns.tolist())

Reordered columns:
['SL', 'LC_Factory', 'LC_No', 'LC_Value', 'LC_Qty', 'Payment_Term', 'Category', 'Buyer', 'Invoice_No', 'Invoice_Date', 'Invoice_Qty', 'Invoice_Value', 'Exp_No', 'Exp_Date', 'Ex_Factory', 'Ex_Factory_Qty', 'Ex_Factory_Date', 'Ex_Factory_No', 'OnBoard_Date', 'Due_Date', 'Discount', 'Commission', 'Shipping_Bill_No', 'Shipping_Bill_Date', 'BL_No', 'BL_Date', 'FCR_No', 'FCR_Date', 'Shipping_Mode', 'Feeder_Vessel_No', 'Mother_Vessel_No', 'Destination', 'Bank', 'Bank_Ref_No', 'Bank_Submit_Date', 'Is_FDBC?', 'Invoice_Status', 'Realized_No', 'Realized_Date', 'Realized_Value', 'Is_Realized?', 'Order_No']


In [14]:
import glob
import re
from datetime import datetime
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter
###################################################################

def clean_illegal_characters(df):
    """Remove illegal Excel characters from all string columns."""
    # Illegal characters in Excel: ASCII 0-31 except tab (9), line feed (10), carriage return (13)
    illegal_chars = ''.join(chr(i) for i in range(32) if i not in (9, 10, 13))
    
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.replace(f'[{re.escape(illegal_chars)}]', '', regex=True)
    
    return df

###################################################################

todaydate = datetime.now().strftime("%Y%m%d")
print(f"Today's date: {todaydate}")

###################################################################
# Save the combined dataframe to the output folder
output_file = os.path.join(OUTPUT_DIR, f"Export_Invoice_Summary_{todaydate}.xlsx")
previous_files = glob.glob(os.path.join(OUTPUT_DIR, "Export_Invoice_Summary_*.xlsx"))
for path in previous_files:
    if path != output_file and os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed previous saved file: {path}")
        except PermissionError:
            print(f"Could not remove file (in use): {path}")

if os.path.exists(output_file):
    try:
        os.remove(output_file)
        print(f"Removed existing file before save: {output_file}")
    except PermissionError:
        print(f"File is in use, will overwrite: {output_file}")

# Clean illegal characters and save dataframe to Excel with table and frozen header
df_clean = clean_illegal_characters(df.copy())

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_clean.to_excel(writer, sheet_name="EIS", index=False, startrow=0)
    workbook = writer.book
    worksheet = writer.sheets["EIS"]
    
    # Freeze the first row (header)
    worksheet.freeze_panes = "A2"
    
    # Create a table named "EIS" covering all data
    last_col = get_column_letter(len(df_clean.columns))
    tab = Table(displayName="EIS", ref=f"A1:{last_col}{len(df_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    # Adjust column widths for better visibility
    for i, col in enumerate(df_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 15

print(f"Saved combined dataframe to {output_file}")
print(f"Table 'EIS' created with frozen header row")
df.head()

Today's date: 20260917
Removed existing file before save: OUTPUT\Export_Invoice_Summary_20260917.xlsx
Saved combined dataframe to OUTPUT\Export_Invoice_Summary_20260917.xlsx
Table 'EIS' created with frozen header row


,SL,LC_Factory,LC_No,LC_Value,LC_Qty,Payment_Term,Category,Buyer,Invoice_No,Invoice_Date,...,Bank,Bank_Ref_No,Bank_Submit_Date,Is_FDBC?,Invoice_Status,Realized_No,Realized_Date,Realized_Value,Is_Realized?,Order_No
0,1,MGSL,MGSL/H&M Kids S2,11273220.28,3616545.0,EOM 63,Woven,H&M,2511591,2025-10-09 00:00:00.000,...,DBBL,FDC-0119-26,2026-01-07,Yes,Incentive Pending,DR-01-26-009,2026-01-08,55.39,Yes,496780-7657
1,2,MGSL,MGSL/H&M-Mens DBL-S.0,11664021.68,2755553.0,EOM+63,Woven,H&M,2412931,2024-09-05 09:36:30.280,...,DBBL,3487-24,2024-10-07,Yes,Incentive Pending,DR-11-24-001,2024-10-08,25.00,Yes,201064-8757
2,3,MGSL,MGSL/PRINCETON SHIRT-8757 S.0,2740870.51,648692.0,EOM+63,Woven,H&M,2405144,2024-03-31 00:00:00.000,...,ABL,529/24,2024-04-30,Yes,Incentive Pending,DR-05-24-047,2024-05-08,400.96,Yes,878905-8757
3,4,MGSL,MGSL/H&M Boys S9,12327213.47,4104494.0,EOM+63,Woven,H&M,2402743,2024-02-14 00:00:00.000,...,ABL,FDBC/0306/24,2024-03-13,Yes,Incentive Pending,DR-03-24-036,2024-03-06,96.48,Yes,872618-7657
4,5,MGSL,MGSL/H&M-Mens DBL-S.02,12435079.87,3145397.0,EOM 63,Woven,H&M,2505213,2025-04-10 14:50:59.230,...,DBBL,FDC-2206-25,2025-06-04,Yes,Incentive Pending,DR-06-25-022,2025-06-04,266.60,Yes,376069-8617


## 4. Extract Ex Factory Pending Report

In [15]:
def build_exfactory_pending_report(source_df, days_pending=7):
    """Return invoices still awaiting ex-factory shipment.

    This function selects rows where `Invoice_Status` equals 'ExFactory Pending' and
    the invoice date is at least `days_pending` days in the past. It drops unwanted
    columns (only if present), sorts by `Invoice_Date`, regenerates a serial column
    and performs a lightweight text cleanup on object/string columns.

    Parameters
    ----------
    source_df : pandas.DataFrame
    days_pending : int, optional
        Minimum number of days since `Invoice_Date` to include (default 7).
    """
    df = source_df.copy()
    today = pd.Timestamp.today().normalize()

    # Ensure Invoice_Date is a datetime dtype so arithmetic works reliably
    if "Invoice_Date" in df.columns:
        df["Invoice_Date"] = pd.to_datetime(df["Invoice_Date"], errors="coerce")

    status_col = "Invoice_Status"
    mask = (
        df[status_col].astype(str).str.strip().eq("ExFactory Pending")
        & df.get("Invoice_Date").notna()
        & ((today - df["Invoice_Date"]).dt.days >= days_pending)
    )

    result = df.loc[mask].copy()

    # Drop columns that are not useful for the pending report (only if present)
    columns_to_drop = {
        "LC_No", "LC_Value", "LC_Qty", "LC_Payment_Term", "Payment_After_ExFactory",
        "Merchandiser", "Port", "Ctn_Qty", "Dozen", "MVessel_No", "Due_Date",
        "FDBC_No", "BankSubmit_Date", "Is_FDBC?", "BL_No", "BL_Date", "FCR_No",
        "FCR_Date", "Carton_Qty", "Discount", "Commission", "Shipping_Bill_No",
        "Shipping_Bill_Date", "Realize_No", "Realize_Date", "Realize_Value",
        "Order_No", "ExFactory_No", "ExFactory",
    }
    drop_list = [c for c in columns_to_drop if c in result.columns]
    if drop_list:
        result = result.drop(columns=drop_list)

    # Sort by invoice date if available
    if "Invoice_Date" in result.columns:
        result = result.sort_values(by="Invoice_Date", ascending=True, na_position="last")

    result = result.reset_index(drop=True)

    # Clean text columns only (keep datetimes and numerics intact)
    text_cols = result.select_dtypes(include=["object", "string"]).columns.tolist()
    if text_cols:
        result[text_cols] = (
            result[text_cols].replace({"nan": "", "NaN": "", "NaT": "", "None": "", "none": ""})
            .fillna("")
        )

    return result

In [16]:
ExFactoryPending = build_exfactory_pending_report(df)

print("ExFactory Pending Table Created Successfully")
print(f"Total Rows: {len(ExFactoryPending)}")
ExFactoryPending.head(3)

ExFactory Pending Table Created Successfully
Total Rows: 84


,SL,LC_Factory,Payment_Term,Category,Buyer,Invoice_No,Invoice_Date,Invoice_Qty,Invoice_Value,Exp_No,...,Mother_Vessel_No,Destination,Bank,Bank_Ref_No,Bank_Submit_Date,Invoice_Status,Realized_No,Realized_Date,Realized_Value,Is_Realized?
0,3515,MGSL,60 Days,Woven,H&M,MGSL/23/2530,2023-04-10,564.0,1804.80,004559,...,,DR (OL-CA),ABL,FDBC-625/23,2023-07-13,ExFactory Pending,DR-07-23-006,2023-07-13,1804.8,Yes
1,24657,MGSL,EOM 63,Woven,H&M,MGSL/23/3013,2023-04-13,89.0,269.67,0042/005481/23,...,,OK (OL-KR),ABL,,NaT,ExFactory Pending,,NaT,NaN,No
2,44645,MGSL,60 Days,Woven,H&M,MGSL/23/3395,2023-05-04,122.0,384.30,006091,...,,PA (PM-PA),ABL,,NaT,ExFactory Pending,,NaT,NaN,No


In [17]:
# Add a clean serial number column starting at 1
if "SL" in ExFactoryPending.columns:
    ExFactoryPending = ExFactoryPending.drop(columns=["SL"])
ExFactoryPending.insert(0, "SL", range(1, len(ExFactoryPending) + 1))

# Clean illegal characters
ExFactoryPending_clean = clean_illegal_characters(ExFactoryPending.copy())

output_file = os.path.join(OUTPUT_DIR, "ExFactoryPending.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    ExFactoryPending_clean.to_excel(writer, sheet_name="ExFactoryPending", index=False)
    worksheet = writer.sheets["ExFactoryPending"]
    
    # Freeze the first row (header)
    worksheet.freeze_panes = "A2"
    
    # Create a table named "ExFactoryPending" covering all data
    last_col = get_column_letter(len(ExFactoryPending_clean.columns))
    tab = Table(displayName="ExFactoryPending", ref=f"A1:{last_col}{len(ExFactoryPending_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium3", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    # Adjust column widths for better visibility
    for i, col in enumerate(ExFactoryPending_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 12

print(f"Saved ExFactoryPending report to {output_file} with sheet name 'ExFactoryPending' and table 'ExFactoryPending'")

Saved ExFactoryPending report to OUTPUT\ExFactoryPending.xlsx with sheet name 'ExFactoryPending' and table 'ExFactoryPending'


## 5. Extract On Board Pending Report

In [18]:
import pandas as pd


def build_onboard_pending_report(source_df):
    """Invoices still awaiting on-board status, 15+ days past ex-factory.
    
    Keeps OnBoard_Date and other key dates for tracking progress.
    Date columns are kept as raw pandas Timestamps (e.g. 2023-04-05 00:00:00)
    with no string formatting applied.
    """
    df = source_df.copy()
    
    # Parse all relevant date columns once — kept as datetime (no strftime)
    date_cols = ["Ex_Factory_Date", "Invoice_Date", "OnBoard_Date"]
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    today = pd.Timestamp.today().normalize()

    # Filter for On Board Pending invoices that are 15+ days past ex-factory
    onboard_pending = df[
        (df["Invoice_Status"] == "On Board Pending")
        & ((today - df["Ex_Factory_Date"]).dt.days >= 15)
    ].copy()

    # Drop columns that are not useful for the pending report (only if present)
    columns_to_drop = [
        "LC_No", "LC_Value", "LC_Qty", "Payment_Term", 
        "Exp_No", "Exp_Date", "Ex_Factory", "Ex_Factory_Qty", "Ex_Factory_No",
        "Due_Date", "Feeder_Vessel_No", "Mother_Vessel_No",
        "Shipping_Bill_No", "Shipping_Bill_Date", "BL_No", "BL_Date", 
        "FCR_No", "FCR_Date", "Bank", "Bank_Ref_No", "Bank_Submit_Date", 
        "Is_FDBC?", "Realized_No", "Realized_Date", "Realized_Value", "Is_Realized?","Order_No"
    ]
    drop_list = [c for c in columns_to_drop if c in onboard_pending.columns]
    if drop_list:
        onboard_pending = onboard_pending.drop(columns=drop_list)

    # Sort by invoice date if available
    if "Invoice_Date" in onboard_pending.columns:
        onboard_pending = onboard_pending.sort_values(by="Invoice_Date", ascending=True, na_position="last")

    onboard_pending = onboard_pending.reset_index(drop=True)
    
    # Clean text columns only (keep datetimes and numerics intact)
    text_cols = onboard_pending.select_dtypes(include=["object", "string"]).columns.tolist()
    if text_cols:
        onboard_pending[text_cols] = (
            onboard_pending[text_cols].replace({"nan": "", "NaN": "", "NaT": "", "None": "", "none": ""})
            .fillna("")
        )

    return onboard_pending

In [19]:
OnBoardPending = build_onboard_pending_report(df)

print("On Board Pending Table Created Successfully")
print(f"Total Rows: {len(OnBoardPending)}")
OnBoardPending.head(3)

On Board Pending Table Created Successfully
Total Rows: 402


,SL,LC_Factory,Category,Buyer,Invoice_No,Invoice_Date,Invoice_Qty,Invoice_Value,Ex_Factory_Date,OnBoard_Date,Discount,Commission,Shipping_Mode,Destination,Invoice_Status
0,8723,MGSL,Woven,H&M,MGSL/23/2753,2023-04-05,85.0,463.25,2023-04-13 10:37:34,NaT,0.0,0.0,Sea,PA (PM-PA),On Board Pending
1,24068,MGSL,Woven,H&M,MGSL/23/2753,2023-04-05,85.0,463.25,2023-04-13 10:37:34,NaT,0.0,0.0,Sea,PA (PM-PA),On Board Pending
2,12223,MGSL,Woven,H&M,MGSL/23/2667,2023-04-08,3504.0,11212.80,2023-04-13 10:37:34,NaT,0.0,0.0,Sea,LD (OL-MX),On Board Pending


In [20]:
# Add a clean serial number column starting at 1
if "SL" in OnBoardPending.columns:
    OnBoardPending = OnBoardPending.drop(columns=["SL"])
OnBoardPending.insert(0, "SL", range(1, len(OnBoardPending) + 1))

# Clean illegal characters
OnBoardPending_clean = clean_illegal_characters(OnBoardPending.copy())

output_file = os.path.join(OUTPUT_DIR, "OnBoardPending.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    OnBoardPending_clean.to_excel(writer, sheet_name="OnBoardPending", index=False)
    worksheet = writer.sheets["OnBoardPending"]
    
    # Freeze the first row (header)
    worksheet.freeze_panes = "A2"
    
    # Create a table named "OnBoardPending" covering all data
    last_col = get_column_letter(len(OnBoardPending_clean.columns))
    tab = Table(displayName="OnBoardPending", ref=f"A1:{last_col}{len(OnBoardPending_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium4", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    # Adjust column widths for better visibility
    for i, col in enumerate(OnBoardPending_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 12

print(f"Saved OnBoardPending report to {output_file} with sheet name 'OnBoardPending' and table 'OnBoardPending'")

Saved OnBoardPending report to OUTPUT\OnBoardPending.xlsx with sheet name 'OnBoardPending' and table 'OnBoardPending'


## 6. Extract Bank Submit Pending Report

In [21]:
def build_bank_submit_pending_report(source_df):
    """Invoices still awaiting bank submission, 7+ days past invoice date.
    
    Keeps Bank_Submit_Date and other key dates for tracking progress.
    Date columns are kept as raw pandas Timestamps with no string formatting.
    """
    df = source_df.copy()
    
    # Parse all relevant date columns once — kept as datetime (no strftime)
    date_cols = ["Invoice_Date", "Bank_Submit_Date", "Ex_Factory_Date"]
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    today = pd.Timestamp.today().normalize()
    
    # Filter for Bank Submit Pending invoices that are 7+ days past invoice date
    mask = (
        df["Invoice_Status"].astype(str).str.strip().eq("Bank Submit Pending")
        & df["Invoice_Date"].notna()
        & ((today - df["Invoice_Date"]).dt.days >= 7)
    )
    bank_submit_pending = df.loc[mask].copy()

    # Drop columns that are not useful for the pending report (only if present)
    columns_to_drop = [
        "LC_No", "LC_Value", "LC_Qty", "Payment_Term",
        "Exp_No", "Exp_Date", "Ex_Factory", "Ex_Factory_Qty", "Ex_Factory_No",
        "Due_Date", "Feeder_Vessel_No", "Mother_Vessel_No",
        "OnBoard_Date", "Shipping_Bill_No", "Shipping_Bill_Date", "BL_No", "BL_Date", 
        "FCR_No", "FCR_Date", "Is_FDBC?", "Realized_No", "Realized_Date", "Realized_Value", "Is_Realized?",
        "Order_No"
    ]
    drop_list = [c for c in columns_to_drop if c in bank_submit_pending.columns]
    if drop_list:
        bank_submit_pending = bank_submit_pending.drop(columns=drop_list)

    # Sort by invoice date if available
    if "Invoice_Date" in bank_submit_pending.columns:
        bank_submit_pending = bank_submit_pending.sort_values(by="Invoice_Date", ascending=True, na_position="last")

    bank_submit_pending = bank_submit_pending.reset_index(drop=True)

    # Clean text columns only (keep datetimes and numerics intact)
    text_cols = bank_submit_pending.select_dtypes(include=["object", "string"]).columns.tolist()
    if text_cols:
        bank_submit_pending[text_cols] = (
            bank_submit_pending[text_cols].replace({"nan": "", "NaN": "", "NaT": "", "None": "", "none": ""})
            .fillna("")
        )

    return bank_submit_pending

In [22]:
BankSubmitPending = build_bank_submit_pending_report(df)

print("Bank Submit Pending Table Created Successfully")
print(f"Total Rows: {len(BankSubmitPending)}")
BankSubmitPending.head(3)

Bank Submit Pending Table Created Successfully
Total Rows: 2833


,SL,LC_Factory,Category,Buyer,Invoice_No,Invoice_Date,Invoice_Qty,Invoice_Value,Ex_Factory_Date,Discount,Commission,Shipping_Mode,Destination,Bank,Bank_Ref_No,Bank_Submit_Date,Invoice_Status
0,34090,MGSL,Woven,H&M,MGSL/23/1893,2023-03-01,9495.0,30004.20,2023-03-09 12:28:30,0.0,0.0,Sea,PL (PMEEU),ABL,,NaT,Bank Submit Pending
1,17718,MGSL,Woven,H&M,MGSL/23/2091,2023-03-10,1934.0,6362.86,2023-03-12 12:32:16,0.0,0.0,Sea,SE (PMSCA),ABL,,NaT,Bank Submit Pending
2,48022,MGSL,Woven,H&M,MGSL/23/2099,2023-03-10,260.0,855.40,2023-03-16 12:36:35,0.0,0.0,Sea,IX (PM-IX),ABL,,NaT,Bank Submit Pending


In [23]:
# Add a clean serial number column starting at 1
if "SL" in BankSubmitPending.columns:
    BankSubmitPending = BankSubmitPending.drop(columns=["SL"])
BankSubmitPending.insert(0, "SL", range(1, len(BankSubmitPending) + 1))

# Clean illegal characters
BankSubmitPending_clean = clean_illegal_characters(BankSubmitPending.copy())

output_file = os.path.join(OUTPUT_DIR, "BankSubmitPending.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    BankSubmitPending_clean.to_excel(writer, sheet_name="BankSubmitPending", index=False)
    worksheet = writer.sheets["BankSubmitPending"]
    
    # Freeze the first row (header)
    worksheet.freeze_panes = "A2"
    
    # Create a table named "BankSubmitPending" covering all data
    last_col = get_column_letter(len(BankSubmitPending_clean.columns))
    tab = Table(displayName="BankSubmitPending", ref=f"A1:{last_col}{len(BankSubmitPending_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium5", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    # Adjust column widths for better visibility
    for i, col in enumerate(BankSubmitPending_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 12

print(f"Saved BankSubmitPending report to {output_file} with sheet name 'BankSubmitPending' and table 'BankSubmitPending'")

Saved BankSubmitPending report to OUTPUT\BankSubmitPending.xlsx with sheet name 'BankSubmitPending' and table 'BankSubmitPending'


## 7. Extract Realize Pending Report

In [24]:
import numpy as np
import pandas as pd

def build_realize_pending_report(source_df):
    """
    Build a Realize Pending report for invoices awaiting realization.
    
    Filters invoices with Invoice_Status = "Realize Pending"
    Keeps all key tracking columns including dates and bank reference.
    
    """
    
    df = source_df.copy()
    
    # Filter for Realize Pending invoices
    if "Invoice_Status" not in df.columns:
        raise KeyError("Required column 'Invoice_Status' was not found.")
    
    realize_report = df[
        df["Invoice_Status"]
        .astype("string")
        .str.strip()
        .eq("Realize Pending")
    ].copy()
    
    # Parse date columns for proper sorting
    date_cols = ["Invoice_Date", "Bank_Submit_Date", "Realized_Date", "Ex_Factory_Date"]
    for col in date_cols:
        if col in realize_report.columns:
            realize_report[col] = pd.to_datetime(realize_report[col], errors="coerce")
    
    # Drop less relevant columns for realize tracking (only if present)
    columns_to_drop = [
        "LC_No", "LC_Value", "LC_Qty", "Payment_Term",
        "Exp_No", "Exp_Date", "Ex_Factory_No",
        "Due_Date", "Feeder_Vessel_No", "Mother_Vessel_No",
        "Shipping_Bill_No", "Shipping_Bill_Date", "BL_No", "BL_Date",
        "FCR_No", "FCR_Date", "Order_No"
    ]
    drop_list = [c for c in columns_to_drop if c in realize_report.columns]
    if drop_list:
        realize_report = realize_report.drop(columns=drop_list)
    
    # Sort by invoice date
    if "Invoice_Date" in realize_report.columns:
        realize_report = realize_report.sort_values(by="Invoice_Date", ascending=True, na_position="last")
    
    realize_report = realize_report.reset_index(drop=True)
    
    # Clean text columns
    text_cols = realize_report.select_dtypes(include=["object", "string"]).columns.tolist()
    if text_cols:
        realize_report[text_cols] = (
            realize_report[text_cols].replace({"nan": "", "NaN": "", "NaT": "", "None": "", "none": ""})
            .fillna("")
        )
    
    return realize_report


# ============================================================
# Generate Realize Pending Report
# ============================================================

RealizePending = build_realize_pending_report(df)

print("Realize Pending Table Created Successfully")
print(f"Total Rows: {len(RealizePending)}")
RealizePending.head(3)

Realize Pending Table Created Successfully
Total Rows: 122


,SL,LC_Factory,Category,Buyer,Invoice_No,Invoice_Date,Invoice_Qty,Invoice_Value,Ex_Factory,Ex_Factory_Qty,...,Destination,Bank,Bank_Ref_No,Bank_Submit_Date,Is_FDBC?,Invoice_Status,Realized_No,Realized_Date,Realized_Value,Is_Realized?
0,4004,MGKFL,Sweater,DK Company,2310036,2023-04-18,500.0,4128.00,MFSL,500.0,...,Denmark,DBBL,1647/23,2023-05-11,Yes,Realize Pending,,NaT,NaN,No
1,35163,MFSL,Sweater,Hellenic Trading AB,2310069,2023-04-20,7550.0,42909.25,MFSL,7550.0,...,Felixtowe,ABL,FDBC/0471/23,2023-05-28,Yes,Realize Pending,,NaT,NaN,No
2,39107,MGKFL,Sweater,WEATHERPROOF VINTAGE,2310067,2023-05-23,9840.0,68388.00,MGKFL,9420.0,...,USA,DBBL,FDBC-2429/23,2023-07-24,Yes,Realize Pending,,NaT,NaN,No


In [25]:
exclude_bank_refs = [
    "FDBC-278/24",
    "302/24",
    "0976/24",
    "0891/24",
    "0938/24",
    "040/24",
]

# Tonni: MGL company excluding specific bank refs
Tonni = RealizePending[
    (RealizePending["LC_Factory"] == "MGL")
    & (~RealizePending["Bank_Ref_No"].isin(exclude_bank_refs))
].copy()

# Nowshin: All Sweater category
Nowshin = RealizePending[RealizePending["Category"] == "Sweater"].copy()

# Arif: MGSL company
Arif = RealizePending[RealizePending["LC_Factory"] == "MGSL"].copy()

# Manzarul: MGNSL company
Manzarul = RealizePending[RealizePending["LC_Factory"] == "MGNSL"].copy()

print(f"Tonni: {len(Tonni)} rows")
print(f"Nowshin: {len(Nowshin)} rows")
print(f"Arif: {len(Arif)} rows")
print(f"Manzarul: {len(Manzarul)} rows")

Tonni: 10 rows
Nowshin: 76 rows
Arif: 2 rows
Manzarul: 5 rows


In [26]:
# Export Realize Pending reports with professional formatting

# Main RealizePending report
RealizePending_clean = clean_illegal_characters(RealizePending.copy())
output_file = os.path.join(REALIZE_REPORTS_DIR, "RealizePending.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    RealizePending_clean.to_excel(writer, sheet_name="RealizePending", index=False)
    worksheet = writer.sheets["RealizePending"]
    
    # Freeze the first row (header)
    worksheet.freeze_panes = "A2"
    
    # Create a table named "RealizePending" covering all data
    last_col = get_column_letter(len(RealizePending_clean.columns))
    tab = Table(displayName="RealizePending", ref=f"A1:{last_col}{len(RealizePending_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    # Adjust column widths for better visibility
    for i, col in enumerate(RealizePending_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 12

print(f"Saved RealizePending report to {output_file}")

# Nowshin report (Sweater category)
Nowshin_clean = clean_illegal_characters(Nowshin.copy())
output_file = os.path.join(REALIZE_REPORTS_DIR, "Nowshin.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    Nowshin_clean.to_excel(writer, sheet_name="Sweater", index=False)
    worksheet = writer.sheets["Sweater"]
    
    worksheet.freeze_panes = "A2"
    
    last_col = get_column_letter(len(Nowshin_clean.columns))
    tab = Table(displayName="Nowshin", ref=f"A1:{last_col}{len(Nowshin_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium3", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    for i, col in enumerate(Nowshin_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 12

print(f"Saved Nowshin report to {output_file}")

# Tonni report (MGL company excluding specific bank refs)
Tonni_clean = clean_illegal_characters(Tonni.copy())
output_file = os.path.join(REALIZE_REPORTS_DIR, "Tonni.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    Tonni_clean.to_excel(writer, sheet_name="MGL", index=False)
    worksheet = writer.sheets["MGL"]
    
    worksheet.freeze_panes = "A2"
    
    last_col = get_column_letter(len(Tonni_clean.columns))
    tab = Table(displayName="Tonni", ref=f"A1:{last_col}{len(Tonni_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium4", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    for i, col in enumerate(Tonni_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 12

print(f"Saved Tonni report to {output_file}")

# Arif report (MGSL company)
Arif_clean = clean_illegal_characters(Arif.copy())
output_file = os.path.join(REALIZE_REPORTS_DIR, "Arif.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    Arif_clean.to_excel(writer, sheet_name="MGSL", index=False)
    worksheet = writer.sheets["MGSL"]
    
    worksheet.freeze_panes = "A2"
    
    last_col = get_column_letter(len(Arif_clean.columns))
    tab = Table(displayName="Arif", ref=f"A1:{last_col}{len(Arif_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium5", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    for i, col in enumerate(Arif_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 12

print(f"Saved Arif report to {output_file}")

# Manzarul report (MGNSL company)
Manzarul_clean = clean_illegal_characters(Manzarul.copy())
output_file = os.path.join(REALIZE_REPORTS_DIR, "Manzarul.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    Manzarul_clean.to_excel(writer, sheet_name="MGNSL", index=False)
    worksheet = writer.sheets["MGNSL"]
    
    worksheet.freeze_panes = "A2"
    
    last_col = get_column_letter(len(Manzarul_clean.columns))
    tab = Table(displayName="Manzarul", ref=f"A1:{last_col}{len(Manzarul_clean) + 1}")
    style = TableStyleInfo(name="TableStyleMedium6", showFirstColumn=False,
                          showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    for i, col in enumerate(Manzarul_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 12

print(f"Saved Manzarul report to {output_file}")
print("\n✓ All Realize Pending reports exported with professional formatting")

Saved RealizePending report to OUTPUT\RealizeReports\RealizePending.xlsx
Saved Nowshin report to OUTPUT\RealizeReports\Nowshin.xlsx
Saved Tonni report to OUTPUT\RealizeReports\Tonni.xlsx
Saved Arif report to OUTPUT\RealizeReports\Arif.xlsx
Saved Manzarul report to OUTPUT\RealizeReports\Manzarul.xlsx

✓ All Realize Pending reports exported with professional formatting


## 8. Export Data Validation

In [27]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['SL', 'LC_Factory', 'LC_No', 'LC_Value', 'LC_Qty', 'Payment_Term',
       'Category', 'Buyer', 'Invoice_No', 'Invoice_Date', 'Invoice_Qty',
       'Invoice_Value', 'Exp_No', 'Exp_Date', 'Ex_Factory', 'Ex_Factory_Qty',
       'Ex_Factory_Date', 'Ex_Factory_No', 'OnBoard_Date', 'Due_Date',
       'Discount', 'Commission', 'Shipping_Bill_No', 'Shipping_Bill_Date',
       'BL_No', 'BL_Date', 'FCR_No', 'FCR_Date', 'Shipping_Mode',
       'Feeder_Vessel_No', 'Mother_Vessel_No', 'Destination', 'Bank',
       'Bank_Ref_No', 'Bank_Submit_Date', 'Is_FDBC?', 'Invoice_Status',
       'Realized_No', 'Realized_Date', 'Realized_Value', 'Is_Realized?',
       'Order_No'],
      dtype='str')>

In [28]:
import os
import re
import numpy as np
import pandas as pd
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ============================================================================
# 1. HELPER FUNCTIONS
# ============================================================================

def clean_illegal_characters(df):
    """Remove non-printable / illegal characters not supported by Excel."""
    illegal_char_regex = re.compile(r"[\000-\010]|[\013-\014]|[\016-\037]")
    return df.apply(
        lambda col: col.map(
            lambda val: illegal_char_regex.sub("", str(val))
            if isinstance(val, str)
            else val
        )
        if col.dtype == "object"
        else col
    )


# ============================================================================
# 2. DATA VALIDATION REPORT FUNCTION
# ============================================================================

def build_data_validation_report(source_df):
    """
    Build a professional data validation report with column-wise validation rules.
    
    Parameters
    ----------
    source_df : pd.DataFrame
        The source dataframe with raw invoice data
        
    Returns
    -------
    pd.DataFrame
        Dataframe with added validation columns in correct sequence and order
    """
    df = source_df.copy()
    
    # SECTION 1: PREPARE DATE COLUMNS
    date_columns = [
        "Invoice_Date", "Exp_Date", "Ex_Factory_Date", "OnBoard_Date",
        "Shipping_Bill_Date", "BL_Date", "FCR_Date", "Bank_Submit_Date",
        "Realized_Date"
    ]
    
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    # SECTION 2: COLUMN-WISE VALIDATION RULES
    
    # ---- Rule 1: IsExpDateWrong? ----
    if "Invoice_Date" in df.columns and "Exp_Date" in df.columns:
        df["IsExpDateWrong?"] = np.where(
            df["Invoice_Date"].isna(),
            "Problem",
            np.where(
                df["Exp_Date"].isna(),
                "",
                np.where(
                    df["Invoice_Date"] <= df["Exp_Date"],
                    "OK",
                    "Problem"
                )
            )
        )
    
    # ---- Rule 2: LessOrHighShipmentQty ----
    if "Invoice_Qty" in df.columns and "Ex_Factory_Qty" in df.columns:
        df["LessOrHighShipmentQty"] = np.where(
            df["Invoice_Qty"].isna(),
            "Problem",
            np.where(
                df["Ex_Factory_Qty"].isna(),
                df["Invoice_Qty"],
                df["Invoice_Qty"] - df["Ex_Factory_Qty"]
            )
        )
    
    # ---- Rule 3: IsExFactoryDateWrong? ----
    if "Invoice_Date" in df.columns and "Ex_Factory_Date" in df.columns:
        df["IsExFactoryDateWrong?"] = np.where(
            df["Invoice_Date"].isna(),
            "Problem",
            np.where(
                df["Ex_Factory_Date"].isna(),
                "",
                np.where(
                    df["Invoice_Date"] <= df["Ex_Factory_Date"],
                    "OK",
                    "Problem"
                )
            )
        )
    
    # ---- Rule 4: IsOnBoardDateWrong? ----
    if "Ex_Factory_Date" in df.columns and "OnBoard_Date" in df.columns:
        df["IsOnBoardDateWrong?"] = np.where(
            df["Ex_Factory_Date"].isna(),
            "",
            np.where(
                df["OnBoard_Date"].isna(),
                "",
                np.where(
                    df["Ex_Factory_Date"] <= df["OnBoard_Date"],
                    "OK",
                    "Problem"
                )
            )
        )
    
    # ---- Rule 5: IsBL_DateWrong? ----
    if "OnBoard_Date" in df.columns and "BL_Date" in df.columns:
        df["IsBL_DateWrong?"] = np.where(
            df["OnBoard_Date"].isna(),
            "",
            np.where(
                df["BL_Date"].isna(),
                "Need BL Date",
                np.where(
                    df["OnBoard_Date"] > df["BL_Date"],
                    "Problem",
                    "OK"
                )
            )
        )
    
    # ---- Rule 6: IsFCR_DateWrong? ----
    if "OnBoard_Date" in df.columns and "FCR_Date" in df.columns:
        df["IsFCR_DateWrong?"] = np.where(
            df["OnBoard_Date"].isna(),
            "",
            np.where(
                df["FCR_Date"].isna(),
                "Need FCR Date",
                np.where(
                    df["OnBoard_Date"] > df["FCR_Date"],
                    "Problem",
                    "OK"
                )
            )
        )
    
    # ---- Rule 7: LessOrHighRealizedAmount ----
    if "Invoice_Value" in df.columns and "Realized_Value" in df.columns:
        df["LessOrHighRealizedAmount"] = np.where(
            df["Invoice_Value"].isna(),
            "Problem",
            np.where(
                df["Realized_Value"].isna(),
                "",
                df["Invoice_Value"] - df["Realized_Value"]
            )
        )
    
    # ---- Rule 8: RealizedAmountStatus ----
    if "Invoice_Value" in df.columns and "Realized_Value" in df.columns:
        df["RealizedAmountStatus"] = np.where(
            df["Invoice_Value"].isna(),
            "Problem",
            np.where(
                df["Realized_Value"].isna(),
                "",
                np.where(
                    (df["Invoice_Value"] - df["Realized_Value"]) > 0,
                    "Less Value Realized",
                    "Over Value Realized"
                )
            )
        )
    
    # SECTION 3: SELECT AND REORDER COLUMNS
    desired_columns = [
        "SL",
        "LC_Factory", "LC_No", "LC_Value", "LC_Qty", "Category", "Buyer",
        "Invoice_No", "Invoice_Date", "Invoice_Qty", "Invoice_Value",
        "Exp_No", "Exp_Date", "IsExpDateWrong?",
        "Ex_Factory", "Ex_Factory_Qty", "LessOrHighShipmentQty",
        "Ex_Factory_Date", "IsExFactoryDateWrong?", "Ex_Factory_No",
        "OnBoard_Date", "IsOnBoardDateWrong?",
        "Shipping_Bill_No", "Shipping_Bill_Date", "BL_No", "BL_Date", "IsBL_DateWrong?",
        "FCR_No", "FCR_Date", "IsFCR_DateWrong?",
        "Bank", "Bank_Ref_No", "Bank_Submit_Date", "Is_FDBC?",
        "Invoice_Status", "Realized_No", "Realized_Date", "Realized_Value", "Is_Realized?",
        "LessOrHighRealizedAmount", "RealizedAmountStatus"
    ]
    
    # Keep only existing columns in order
    desired_columns = [col for col in desired_columns if col in df.columns]
    df = df[desired_columns]
    
    return df


# ============================================================================
# 3. RUN VALIDATION AND EXPORT WITH FORMATTING
# ============================================================================

# Directory setup
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_file = os.path.join(OUTPUT_DIR, "DataValidationReport.xlsx")

# 1. Call validation function
validation_df = build_data_validation_report(df_clean)
print("Data Validation Report Created Successfully")

# 2. Clean characters before exporting to openpyxl
validation_df_clean = clean_illegal_characters(validation_df.copy())

# 3. Export to Excel with styling
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    validation_df_clean.to_excel(writer, sheet_name="Validation", index=False)
    workbook = writer.book
    worksheet = writer.sheets["Validation"]
    
    # Freeze the header row
    worksheet.freeze_panes = "A2"
    
    # Define validation columns to highlight (yellow background + red bold text)
    highlight_cols = {
        "IsExpDateWrong?",
        "LessOrHighShipmentQty",
        "IsExFactoryDateWrong?",
        "IsOnBoardDateWrong?",
        "IsBL_DateWrong?",
        "IsFCR_DateWrong?",
        "LessOrHighRealizedAmount",
        "RealizedAmountStatus",
    }
    
    yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    red_font = Font(bold=True, color="FF0000")
    
    # Apply styling to specified header columns
    for col_num, col_name in enumerate(validation_df_clean.columns, 1):
        cell = worksheet.cell(row=1, column=col_num)
        if col_name in highlight_cols:
            cell.fill = yellow_fill
            cell.font = red_font
    
    # Format as an Excel table
    last_col = get_column_letter(len(validation_df_clean.columns))
    tab = Table(displayName="Validation", ref=f"A1:{last_col}{len(validation_df_clean) + 1}")
    style = TableStyleInfo(
        name="TableStyleMedium10",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False
    )
    tab.tableStyleInfo = style
    worksheet.add_table(tab)
    
    # Adjust column widths
    for i, col in enumerate(validation_df_clean.columns, 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 16

print(f"Saved Data Validation Report to {output_file}")
print("Validation columns highlighted with yellow background and red text")
print(f"Total rows: {len(validation_df_clean)}")

Data Validation Report Created Successfully
Saved Data Validation Report to ./output\DataValidationReport.xlsx
Validation columns highlighted with yellow background and red text
Total rows: 52003
